In [1]:
import torch
import torch.nn as nn
from d2l import torch as d2l
import torch.nn.functional as F
import torch.nn as nn

In [2]:
import time

In [3]:

scale=0.1
W1=torch.randn([20,1,3,3])*scale
b1=torch.zeros(20)
W2=torch.randn([50,20,5,5])*scale
b2=torch.zeros(50)
W3=torch.randn([800,128])*scale
b3=torch.zeros(128)
W4=torch.randn([128,10])*scale
b4=torch.zeros(10)
init_weight=[W1,b1,W2,b2,W3,b3,W4,b4]

def net(X,params):
    H1= F.conv2d(input=X,weight=params[0],bias=params[1])
    H1_activated=F.relu(H1)
    H2=F.avg_pool2d(H1_activated,kernel_size=[2,2],stride=[2,2])
    H3=F.conv2d(input=H2,weight=params[2],bias=params[3])
    H3_activated=F.relu(H3)
    H4=F.avg_pool2d(H3_activated,kernel_size=[2,2],stride=[2,2])
    H4=H4.reshape(H4.shape[0],-1)
    H5=torch.mm(H4,params[4])+params[5]
    H5_activated=F.relu(H5)
    H6=torch.mm(H5_activated,params[6])+params[7]
    return H6

In [4]:
class Timer:
    def __init__(self,description=""):
        self.startTime=time.time()
        self.description=description 
        return

    def __call__(self):
        return print(f"Run Time of {self.description}：{time.time()-self.startTime}\n")

In [5]:
def get_all_gpus():
    devices=[torch.device('cuda:0'),torch.device('cuda:1'),torch.device('cuda:2')]
    return devices

def dataParallel(data,devices):
    return torch.nn.parallel.scatter(data,devices)

def allReduece(flats):
    total = flats[0].clone()                      # 1 次分配
    for f in flats[1:]:
        total.add_(f.to(total.device))            # 2 次 .to()，累加到 total
    for f in flats:
        f.copy_(total.to(f.device))               # 3 次 .to()，但目标 buffer 已存在，无分配

def get_params_device(params,device):
    new_params=[param.to(device) for param in params]
    for param in new_params:
        param.requires_grad_()
    return new_params

batch_size=512
train_iter,test_iter=d2l.load_data_fashion_mnist(batch_size)
loss = nn.CrossEntropyLoss(reduction='none')


In [6]:

def train_one_ring(net,loss,param,train_iter,test_iter,epochs,devices,lr=0.5):
    params=[]
    for device in devices:
        new_param=get_params_device(param,device)
        params.append(new_param)

    timer=Timer("all epoch")
    for epoch in range(epochs):
        for X,y in train_iter:
            X_partial=dataParallel(X,devices)
            y_partial=dataParallel(y,devices)
            for xp,yp,para in zip(X_partial,y_partial,params):
                l=loss(net(xp,params=para),yp).sum()
                l.backward()
            with torch.no_grad():
                # 1. 每卡把所有梯度拼成一个 1D buffer
                flats = []
                for para in params:
                    flat = torch.cat([p.grad.reshape(-1) for p in para]).contiguous()
                    flats.append(flat)

                # 2. 只调用一次 allReduce（8 次 -> 1 次）
                allReduece(flats)

                # 3. 拆回每个参数的 grad
                for c, para in enumerate(params):
                    off = 0
                    for p in para:
                        n = p.numel()
                        p.grad = flats[c][off:off+n].reshape(p.shape)
                        off += n

                # 4. 更新
                for c in range(len(devices)):
                    d2l.sgd(params[c], lr, X.shape[0])
        accuracy=d2l.evaluate_accuracy_gpu(
            lambda x: net(x, params[0]), test_iter, devices[0])
        print(f"epoch:{epoch}: accuracy:{accuracy:.4f}")
    timer()
    return params

In [7]:
train_one_ring(net,loss,init_weight,train_iter,test_iter,epochs=10,devices=get_all_gpus())

epoch:0: accuracy:0.6628
epoch:1: accuracy:0.7508
epoch:2: accuracy:0.6898
epoch:3: accuracy:0.6165
epoch:4: accuracy:0.7415
epoch:5: accuracy:0.7624
epoch:6: accuracy:0.7036
epoch:7: accuracy:0.7691
epoch:8: accuracy:0.7031
epoch:9: accuracy:0.8011
Run Time of all epoch：392.4447932243347



[[tensor([[[[ 0.3914, -0.0316,  0.0645],
            [ 0.1744, -0.1684, -0.0303],
            [ 0.0721, -0.1041, -0.1899]]],
  
  
          [[[-0.2996, -0.1347,  0.1727],
            [-0.0713,  0.0020, -0.0688],
            [-0.0850, -0.2686,  0.3329]]],
  
  
          [[[ 0.1939,  0.3673,  0.1537],
            [ 0.2981,  0.4480,  0.1728],
            [-0.0377,  0.2983,  0.1049]]],
  
  
          [[[ 0.0581, -0.0768, -0.1274],
            [ 0.0157, -0.0600, -0.0841],
            [-0.1260,  0.0620, -0.0214]]],
  
  
          [[[ 0.2279,  0.2116, -0.4556],
            [ 0.3212,  0.0306, -0.4191],
            [ 0.2714, -0.0148, -0.2376]]],
  
  
          [[[-0.2435,  0.1579,  0.1475],
            [-0.1796,  0.0385,  0.0242],
            [ 0.0273,  0.2135,  0.1439]]],
  
  
          [[[ 0.0077,  0.0574,  0.0627],
            [-0.0987, -0.0530, -0.1003],
            [-0.0304, -0.0974, -0.0066]]],
  
  
          [[[-0.0529, -0.0592,  0.0230],
            [-0.0334,  0.0593, -0.0725],
 

In [8]:
for i in range(4):
    for j in range(4):
        if i != j:
            print(i, j, torch.cuda.can_device_access_peer(i, j))

0 1 True
0 2 True


AssertionError: Invalid peer device id